# Lab 2: Text and Labels as Measurements


## How this Colab lab works

Use one shared team copy. One person is the **driver** and runs code; the other is the **statistical navigator** and checks the unit, denominator, split, and claim. Switch roles at the marked handoff.

1. Record the individual prediction before revealing output.
2. Run the instructor example.
3. Modify the supplied code with your partner.
4. Run the supplied structural check.
5. Inspect actual images or text when requested.
6. Write the independent handoff in your own words.

The notebook file is shared; each collaborator's temporary Colab runtime is not. Avoid two people executing different versions simultaneously. Save the notebook before switching drivers.


In [ ]:
# Standard Colab setup - run once
from pathlib import Path
import hashlib
import json
import random
import sys
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2027
random.seed(SEED)
np.random.seed(SEED)

COURSE_REPO_RAW_URL = 'https://raw.githubusercontent.com/skgallagher/stat-methods-ai-public/agent/week02-colab-preview'
COURSE_DATA_BASE_URL = COURSE_REPO_RAW_URL + '/data/course'
COURSE_DATA_GROUPS = ['cfpb', 'dynasent']
DATA_ROOT = Path('/content/stat_ai_data')

# Local repository runs use the frozen release when present and otherwise the
# synthetic smoke fixture. A fresh Colab downloads verified individual files
# from GitHub - no ZIP upload or Drive mount is required.
LOCAL_RELEASE = Path.cwd() / 'data' / 'course'
LOCAL_SMOKE = Path.cwd() / 'data' / 'smoke'
online_release = False
if (LOCAL_RELEASE / 'manifest.json').exists():
    DATA_ROOT = LOCAL_RELEASE
    data_source = 'local frozen release'
elif LOCAL_SMOKE.exists():
    DATA_ROOT = LOCAL_SMOKE
    data_source = 'local synthetic smoke fixture (development only)'
elif (DATA_ROOT / 'manifest.json').exists():
    cached_manifest = json.loads((DATA_ROOT / 'manifest.json').read_text())
    if 'smoke fixture' in cached_manifest.get('bundle_type', ''):
        data_source = 'existing local synthetic smoke fixture (development only)'
    else:
        cached_files = cached_manifest.get('files', [])
        requested_files = [
            item for item in cached_files
            if Path(item['path']).parts[0] in COURSE_DATA_GROUPS
        ]
        def cached_sha256(path):
            digest = hashlib.sha256()
            with path.open('rb') as stream:
                for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                    digest.update(chunk)
            return digest.hexdigest()
        cache_complete = (
            cached_manifest.get('release_status') == 'student_release'
            and requested_files
            and all(
                (DATA_ROOT / item['path']).exists()
                and cached_sha256(DATA_ROOT / item['path']) == item['sha256']
                for item in requested_files
            )
        )
        if cache_complete:
            data_source = 'existing verified runtime cache'
        else:
            online_release = True
else:
    online_release = True

if online_release:
    helper_target = Path('/content/course_helpers/__init__.py')
    helper_target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        COURSE_REPO_RAW_URL + '/course_helpers/__init__.py', helper_target
    )
    if '/content' not in sys.path:
        sys.path.insert(0, '/content')
    from course_helpers import ensure_course_data
    DATA_ROOT = ensure_course_data(
        COURSE_DATA_BASE_URL,
        DATA_ROOT,
        groups=COURSE_DATA_GROUPS,
    )
    data_source = 'public GitHub student release'

print('Setup complete. Data root:', DATA_ROOT)
print('Data source:', data_source)
print('Requested groups:', COURSE_DATA_GROUPS)


**Lab role:** CFPB provides substantial text for EDA; DynaSent provides repeated label measurements. HW2 develops the DynaSent measurement argument independently.

**Save before leaving:** `lab2_measurement_notes.md` and the rater-summary table.

**Student working file:** `lab_starter.ipynb`. Switch driver/navigator roles when the lab moves from CFPB to DynaSent.

**Reading in hand:** Use the two assigned Shalizi sections for the independence/exchangeability audit and the assigned DynaSent sections for one concrete construction detail.

## Lecture bridge — 3 minutes

For each dataset, name the observational unit, how text entered the dataset, who supplied the label, and one population missing from observation.

### Individual response — before running output

TODO: record your prediction, estimand/unit, or design choice in 1–3 sentences.


## Instructor run — 6 minutes

Load a frozen CFPB extract and display six randomly selected narratives with product, issue, date, and submission channel.

### Individual response — before running output

TODO: record your prediction, estimand/unit, or design choice in 1–3 sentences.


In [ ]:
import pandas as pd
import numpy as np

complaints = pd.read_csv(DATA_ROOT / "cfpb/complaints.csv")
dynasent = pd.read_csv(DATA_ROOT / "dynasent/items.csv")

complaints["n_words"] = complaints["narrative"].fillna("").str.split().str.len()
complaints.sample(6, random_state=2027)[
    ["narrative", "product", "issue", "date_received", "submitted_via"]
]

Together, distinguish random examples from extreme examples. Identify visible
redaction. The CFPB publishes a narrative only when the consumer opts to share
it and after personal-information removal. In the current database, public
narratives occur only for Web submissions, although complaints can enter the
system through other channels.

## Pairs modify — 8 minutes

Build one compact CFPB EDA display with:

- product counts with denominators;
- narrative length overall and for the two largest products;
- four actual narratives selected by a locked rule: two random and two longest.

Each pair chooses one finding and completes: “This changes our planned analysis because …”

The instructor supplies the exact-duplicate summary and the CFPB publication
rule for discussion. Students interpret them but do not build four separate
displays in ten minutes.

## Stop and discuss — 5 minutes

Compare findings. The instructor asks:

- Is a consumer-selected issue a fact, a measurement, or both?
- Why is the narrative subset selected twice—into the complaint system and into publication?
- Which EDA display affects a later modeling choice rather than merely describing the file?

### Individual discussion response

Write your own answer before comparing wording with your partner.

TODO


## Independent handoff — 10 minutes

Turn to the DynaSent sample. Expand the released label distribution to one row per item–rater judgment using the supplied helper, then compute:

- majority label;
- vote pattern (`5-0`, `4-1`, or `3-2`);
- model correctness under the majority label;
- model probability assigned to the rater-proportion outcome.

In [ ]:
from course_helpers import expand_dynasent_raters, summarize_votes

ratings = expand_dynasent_raters(dynasent)
rater_summary = summarize_votes(ratings, dynasent)
rater_summary.to_csv("lab2_rater_summary.csv", index=False)

### Independent handoff

Write this individually before discussing wording with your partner.

TODO


Check that every item expands to exactly five rater rows and that the three class proportions sum to one.

In [ ]:
assert ratings.groupby("item_id").size().eq(5).all()
assert np.allclose(
    rater_summary[["p_negative", "p_neutral", "p_positive"]].sum(axis=1), 1
)

For the first item, compute the Brier score twice using the lecture convention: once against the one-hot majority label and once against the three rater proportions. Then generalize the calculation to every item.

In [ ]:
LABELS = ["negative", "neutral", "positive"]
model_prob = rater_summary[[f"model_p_{label}" for label in LABELS]].to_numpy()
soft_target = rater_summary[[f"p_{label}" for label in LABELS]].to_numpy()
hard_target = np.array([
    [int(label == majority) for label in LABELS]
    for majority in rater_summary["majority_label"]
])
rater_summary["hard_brier"] = ((model_prob - hard_target) ** 2).sum(axis=1)
rater_summary["soft_brier"] = ((model_prob - soft_target) ** 2).sum(axis=1)
assert rater_summary[["hard_brier", "soft_brier"]].ge(0).all().all()
assert rater_summary[["hard_brier", "soft_brier"]].le(2).all().all()

Create one denominator-first table crossing collection round and vote pattern. Write two findings: one about raters and one about collection round. Do not call disagreement “noise” without specifying the construct you believe is being measured, and do not describe the round difference as causal.

## Stop and discuss assumptions — 5 minutes

For each design below, decide separately whether independence and exchangeability
are plausible. Name the design evidence you would need; “there are several
labels” is not evidence.

| Design | Independence? Why? | Exchangeability? Why? |
|---|---|---|
| Five crowd raters use the same prompt and cannot see one another's answers | TODO | TODO |
| One expert and four novices label separately | TODO | TODO |
| A student and two frozen classifiers forecast the same ten sentences | TODO | TODO |

Then complete these two statements:

- Even if independence fails, the observed $\widehat q$ still describes ___ .
- The formula $q_k(1-q_k)/m$ additionally requires ___ .

Be precise about the unit. Ratings from different people on one sentence and
ratings from the same person across several sentences create different possible
dependencies.

### Individual discussion response

Write your own answer before comparing wording with your partner.

TODO


## Exit ticket — 2 minutes

1. Name one observed variable you initially treated as truth but now regard as a
   measurement produced by a protocol.
2. Give one reason several labels might be independent but not exchangeable, or
   exchangeable but not independent.

**HW2 begins here:** reuse the rater expansion and summary code on new items.
Homework distinguishes individual measurements, empirical soft targets,
majority targets, and forecasts; it also asks you to audit the independence and
interchangeability claim for a student and two frozen models.

The lab does **not** establish that collection round caused any performance difference. It gives you the checked item-level table and scoring code needed to make a bounded descriptive argument on new homework items.

### Exit-ticket response

TODO


## Before leaving

- [ ] The notebook runs in order through the required handoff.
- [ ] Denominators, units, and evaluated population are visible.
- [ ] Required image/text cases are displayed rather than merely described.
- [ ] The driver and navigator switched at least once.
- [ ] Each person wrote the independent handoff.
- [ ] The named artifact was saved for the homework or project.
